## 📌 Setup & Dependencies

In [0]:
# Databricks: install yfinance (lightweight) if not already available
%pip install --quiet yfinance

# After a %pip install, Databricks recommends a restart of Python to ensure the environment is refreshed.
dbutils.library.restartPython()

In [0]:
import os, json
from pathlib import Path

import yfinance as yf
import pandas as pd

## 📌 Load Config & Resolve Paths 

In [0]:
# ================================
# 1. Locate the repo root safely
# ================================
REPO_NAME = "magnificent7stocks-etl"   # <-- your repo folder name EXACTLY

cwd = Path(os.getcwd())
repo_root = cwd

while repo_root.name != REPO_NAME and repo_root != repo_root.parent:
    repo_root = repo_root.parent

print("───── REPO PATH VALIDATION ─────")
if repo_root.name == REPO_NAME:
    print(f"[OK] Repo root found: {repo_root}")
else:
    print(f"[ERROR] Could not locate repo '{REPO_NAME}'.")
    print("Current working dir:", cwd)
    raise FileNotFoundError("Repo root could not be found. Check REPO_NAME.")

# ================================
# 2. Validate config folder
# ================================
config_dir = repo_root / "02_config"

print("\n───── CONFIG FOLDER VALIDATION ─────")
print(f"Expected config folder: {config_dir}")

if not config_dir.exists():
    print(f"[ERROR] Config folder does NOT exist: {config_dir}")
    print("Make sure the structure is:")
    print(f"{REPO_NAME}/02_config/")
    raise FileNotFoundError("Config folder missing.")
else:
    print("[OK] Config folder found.")

# ================================
# 3. Validate individual JSON files
# ================================
tickers_path = config_dir / "tickers.json"
api_cfg_path = config_dir / "api_config.json"

print("\n───── FILE VALIDATION ─────")

# --- Tickers file ---
if tickers_path.exists():
    print(f"[OK] tickers.json found at: {tickers_path}")
    with open(tickers_path) as f:
        tickers = json.load(f).get("tickers", [])
else:
    print(f"[WARNING] tickers.json NOT found at: {tickers_path}")
    print("→ Using default tickers.")
    tickers = ["AAPL", "AMZN", "GOOGL", "META", "MSFT", "NVDA", "TSLA"]

# --- API configuration file ---
if api_cfg_path.exists():
    print(f"[OK] api_config.json found at: {api_cfg_path}")
    with open(api_cfg_path) as f:
        api_cfg = json.load(f)
else:
    print(f"[WARNING] api_config.json NOT found at: {api_cfg_path}")
    print("→ Using default API configuration.")
    api_cfg = {"source": "yfinance", "interval": "1h", "period": "1d"}

# ================================
# 4. Extract values with fallbacks
# ================================
interval = api_cfg.get("interval", "1d")
period = api_cfg.get("period", "1mo")

print("\n───── FINAL CONFIG LOADED ─────")
print("Tickers:", tickers)
print("Interval:", interval)
print("Period:", period)
print("================================\n")


## 📌 Ingest Data From Yahoo Finance  
### Why Data Extraction Is Done *Block by Block*

To ensure stable and reliable ingestion of stock market data from the Yahoo Finance API, each ticker is downloaded **individually in separate blocks** instead of using a single loop. This approach is used because:

- **Prevents API throttling**  
  Yahoo Finance often blocks or limits rapid consecutive requests from cloud environments, causing empty or incomplete responses when fetching multiple tickers in a loop.

- **Ensures complete and consistent data**  
  Downloading one ticker at a time avoids partially loaded or corrupted DataFrames that can break ETL processes.

- **Improves debugging clarity**  
  Each block shows whether a specific ticker succeeded or failed, making troubleshooting simpler and more efficient.

- **Avoids schema and concatenation issues**  
  When some tickers return empty results, merging them inside a loop can cause schema mismatches, Pandas errors, or Spark type‑casting problems. Block‑by‑block extraction eliminates this risk.

- **Follows robust ETL patterns**  
  Processing each data source independently increases reliability and prevents one failure from affecting the entire ingestion pipeline.

Stock 1 — AAPL

In [0]:
ticker = tickers[0]  # AAPL
print("Downloading:", ticker)

df_0 = yf.download(ticker, period=period, interval=interval, auto_adjust=False)

print(df_0.head())
print("Rows:", len(df_0))


Stock 2 — AMZN

In [0]:
ticker = tickers[1]  # AMZN
print("Downloading:", ticker)

df_1 = yf.download(ticker, period=period, interval=interval, auto_adjust=False)

print(df_1.head())
print("Rows:", len(df_1))

Stock 3 — GOOGL

In [0]:
ticker = tickers[2]  # GOOGL
print("Downloading:", ticker)

df_2 = yf.download(ticker, period=period, interval=interval, auto_adjust=False)

print(df_2.head())
print("Rows:", len(df_2))


Stock 4 — META

In [0]:
ticker = tickers[3]  # META
print("Downloading:", ticker)

df_3 = yf.download(ticker, period=period, interval=interval, auto_adjust=False)

print(df_3.head())
print("Rows:", len(df_3))


Stock 5 — MSFT

In [0]:
ticker = tickers[4]  # MSFT
print("Downloading:", ticker)

df_4 = yf.download(ticker, period=period, interval=interval, auto_adjust=False)

print(df_4.head())
print("Rows:", len(df_4))

Stock 6 — NVDA

In [0]:
ticker = tickers[5]  # NVDA
print("Downloading:", ticker)

df_5 = yf.download(ticker, period=period, interval=interval, auto_adjust=False)

print(df_5.head())
print("Rows:", len(df_5))

Stock 7 — TSLA

In [0]:
ticker = tickers[6]  # TSLA
print("Downloading:", ticker)

df_6 = yf.download(ticker, period=period, interval=interval, auto_adjust=False)

print(df_6.head())
print("Rows:", len(df_6))